In [16]:
import os
import pandas as pd
import re
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Especifica la ruta de la carpeta
carpeta = './Data/'

# Lista todos los archivos en la carpeta y filtra los que terminan en .csv
archivos_csv = [archivo for archivo in os.listdir(carpeta) if archivo.endswith('.csv')]

# Imprime los nombres de los archivos CSV
for archivo in archivos_csv:
    print(archivo)

# Lista todos los archivos CSV en la carpeta
archivos_csv = [archivo for archivo in os.listdir(carpeta) if archivo.endswith('.csv')]

for archivo in archivos_csv:
    # Crear un nombre válido para la variable eliminando caracteres no permitidos
    nombre_variable = 'df_' + re.sub(r'\W+', '_', os.path.splitext(archivo)[0])
    
    # Construir la ruta completa del archivo
    ruta_completa = os.path.join(carpeta, archivo)
    
    # Leer el CSV y asignarlo a la variable dinámica
    globals()[nombre_variable] = pd.read_csv(ruta_completa)
    
    # Opcional: imprimir el nombre de la variable creada
    print(f"Variable creada: {nombre_variable}")


df_merge_matrixx_head.csv
df_merge_matrixx_MX-QAR-VMDmLNmBOOSTVMAT.csv
MX-QAR-VMDmLNmBOOSTVMAT.csv
PV2D_AdditionalInfo.csv
PV2D_CalculationParameters.csv
PV2D_DataContainer.csv
PV2D_DicomInformation.csv
PV2D_FieldData_filtered.csv
PV2D_Patient.csv
Variable creada: df_df_merge_matrixx_head
Variable creada: df_df_merge_matrixx_MX_QAR_VMDmLNmBOOSTVMAT
Variable creada: df_MX_QAR_VMDmLNmBOOSTVMAT
Variable creada: df_PV2D_AdditionalInfo
Variable creada: df_PV2D_CalculationParameters
Variable creada: df_PV2D_DataContainer
Variable creada: df_PV2D_DicomInformation
Variable creada: df_PV2D_FieldData_filtered
Variable creada: df_PV2D_Patient


In [2]:
df_merge = pd.merge(df_PV2D_FieldData_filtered, df_PV2D_DataContainer, on='Id', how='left')

In [ ]:
df_merge.head()  # Muestra las primeras filas del DataFrame resultante

,Id,Data_ContainedType,Data_CountTotal,Data_Elements,Data_DimensionCount,Data_CountX,Data_CountY,Data_CountZ,Data_ReferenceX,Data_ReferenceY,...,ParentId,Name,Comment,ReportComment,ReportCheckupStateIndex,CreatedAt,CreatedBy,LastChangedAt,LastChangedBy,Discriminator
0,BF6B0782-A9F7-4D89-B795-0005D3657170,System.Double,1024,b'\x00\x01\x00\x00\x00\xff\xff\xff\xff\x01\x00...,2,32,32,0,-118.0945,-118.0945,...,6C28475F-B875-4A65-9556-28073BBF3704,Integral 30/01/2024 11:09:49.32,NaN,NaN,0,2024-01-30 11:15:53.752875,sdestri,2024-01-30 11:18:22.897553,sdestri,NaN
1,8142949D-22C7-4A07-B111-001E83981DE0,System.Double,1024,b'\x00\x01\x00\x00\x00\xff\xff\xff\xff\x01\x00...,2,32,32,0,-118.0945,-118.0945,...,A1F39C55-DC0D-4833-8158-1633FC178AAC,MX-QAR-VMDmLNmBOOSTVMAT,NaN,NaN,0,2025-01-03 17:14:36.482146,sbianchini,2025-01-06 11:59:30.021048,sbianchini,NaN
2,EE8EA67C-FDD3-449A-922D-005AECC5F7C7,System.Double,1024,b'\x00\x01\x00\x00\x00\xff\xff\xff\xff\x01\x00...,2,32,32,0,-118.0945,-118.0945,...,CC4A7E09-7840-4AC3-A793-296B61139E2F,Integral 06/06/2024 17:41:47.31,NaN,NaN,0,2024-06-06 17:50:09.248318,sdestri,2024-06-06 17:54:52.705178,sdestri,NaN
3,E9E3DBA7-6433-49AC-A636-00BB23F70159,System.Double,1024,b'\x00\x01\x00\x00\x00\xff\xff\xff\xff\x01\x00...,2,32,32,0,-118.0945,-118.0945,...,2A223D9F-815B-4B65-A601-CF9D642A96F8,MX RX2A QAR,NaN,NaN,0,2024-10-24 18:18:24.166222,rlapera,2024-10-24 19:13:19.119024,rlapera,NaN
4,867FC52E-04B2-469A-A62E-00C015B59F75,System.Double,1024,b'\x00\x01\x00\x00\x00\xff\xff\xff\xff\x01\x00...,2,32,32,0,-118.0945,-118.0945,...,14920707-059E-46A1-A5CA-CF67F8A5FE94,Integral 10/04/2024 17:20:04.14,NaN,NaN,0,2024-04-10 17:21:27.147519,dtolabin,2024-04-10 17:26:33.511261,dtolabin,NaN


In [4]:
df_merge.shape

(593, 51)

In [5]:
df_merge['Data_CountX'].value_counts().sort_index(ascending=False)
df_merge_matrixx = df_merge[df_merge['Data_CountY']==32]
df_merge_matrixx.shape

(592, 51)

In [6]:
df_merge_matrixx.head().to_csv('./Data/df_merge_matrixx_head.csv', index=False)

In [ ]:
def decode_matrix_from_dataframe(df, dtypes, num_columns=33):
    # Paso 1: Extraer los datos binarios desde la columna Data_Elements
    data_elements_str = df.loc[0, 'Data_Elements']
    
    # Interpretar el string como bytes (eval convierte el string b'...' en bytes reales)
    data_bytes = eval(data_elements_str)
    
    results = {}

    # Paso 2: Probar cada dtype
    for dtype in dtypes:
        try:
            array_flat = np.frombuffer(data_bytes, dtype=dtype)
            total_elements = len(array_flat)
            num_rows = total_elements // num_columns

            if total_elements % num_columns == 0:
                reshaped_array = array_flat.reshape((num_rows, num_columns))
                results[str(dtype)] = {
                    'array': reshaped_array,
                    'shape': reshaped_array.shape,
                    'min': reshaped_array.min(),
                    'max': reshaped_array.max()
                }
            else:
                results[str(dtype)] = f"Incompatible: {total_elements} elementos no divisible por {num_columns} columnas."
        except Exception as e:
            results[str(dtype)] = f"Error: {e}"

    return results


# Load the dataframe
df_matrixx = pd.read_csv('./Data/df_merge_matrixx_MX-QAR-VMDmLNmBOOSTVMAT.csv')
dtypes = [
        np.int8, np.uint8,
        np.int16, np.uint16,
        np.int32, np.uint32,
        np.float16, np.float32, np.float64
    ]
decoded_results = decode_matrix_from_dataframe(df_matrixx, dtypes=dtypes)

# Mostrar resumen de resultados
for dtype, result in decoded_results.items():
    print(f"\n=== Decoding as {dtype} ===")
    if isinstance(result, dict):
        print(f"Shape: {result['shape']}, Min: {result['min']}, Max: {result['max']}")
    else:
        print(result)


In [ ]:
def validar_data_elements(df, posibles_dtypes=None, bins=50):

    count_x = df.Data_CountX 
    count_y = df.Data_CountY

    data_elements_str = df.loc[0, 'Data_Elements']
    
    # Interpretar el string como bytes (eval convierte el string b'...' en bytes reales)
    data_bytes = eval(data_elements_str)

    if posibles_dtypes is None:
        posibles_dtypes = [np.int16, np.uint16, np.int32, np.uint32, np.float32, np.float64]
    
    total_esperado = count_x * count_y
    
    print(f"Esperando {total_esperado} elementos para una matriz {count_x}x{count_y}")
    
    for dtype in posibles_dtypes:
        try:
            array = np.frombuffer(data_bytes, dtype=dtype)
            n = array.size
            print(f"\nProbando dtype {dtype}: elementos detectados = {n}")
            if n == total_esperado:
                matriz = array.reshape((count_y, count_x))  # filas x columnas
                print(f"Match exacto con dtype {dtype}, mostrando imagen y histograma...")
                
                fig, axs = plt.subplots(1, 2, figsize=(12,5))
                
                # Imagen de la matriz
                axs[0].imshow(matriz, cmap='viridis')
                axs[0].set_title(f"Visualización Data_Elements con dtype {dtype}")
                axs[0].set_xlabel('Columnas')
                axs[0].set_ylabel('Filas')
                cbar = plt.colorbar(axs[0].images[0], ax=axs[0])
                cbar.set_label('Valor')
                
                # Histograma
                axs[1].hist(array, bins=bins, color='steelblue', edgecolor='black')
                axs[1].set_title(f"Histograma de valores con dtype {dtype}")
                axs[1].set_xlabel('Valor')
                axs[1].set_ylabel('Frecuencia')
                
                plt.tight_layout()
                plt.show()
            else:
                print(f"No coincide el tamaño esperado ({total_esperado}) con {n} elementos para dtype {dtype}")
        except Exception as e:
            print(f"Error al decodificar con dtype {dtype}: {e}")

# Ejemplo de uso:
validar_data_elements(df_matrixx)


NameError: name 'count_x' is not defined